In [312]:
import pandas as pd
import numpy as np
import re
import difflib
import importlib
import cleaning as cln
importlib.reload(cln)


<module 'cleaning' from 'd:\\DS108\\DS108_Phone-Reccomendation\\cleaning.py'>

### IMPORTING

In [313]:
cps = pd.read_csv(r'cellphones_raw.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 50 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Unnamed                 966 non-null    int64
 1    0                      966 non-null    int64
 2   Name                    966 non-null    str  
 3   Price                   333 non-null    str  
 4   Link                    966 non-null    str  
 5   Kích thước màn hình     873 non-null    str  
 6   Công nghệ màn hình      814 non-null    str  
 7   Camera sau              855 non-null    str  
 8   Camera trước            823 non-null    str  
 9   Chipset                 857 non-null    str  
 10  Công nghệ NFC           762 non-null    str  
 11  Bộ nhớ trong            915 non-null    str  
 12  Thẻ SIM                 696 non-null    str  
 13  Hệ điều hành            762 non-null    str  
 14  Độ phân giải màn hình   659 non-null    str  
 15  Tính năng màn hình      732 non-nu

In [314]:
cps = cps.drop(columns=['Unnamed', ' 0'])

In [315]:
df = cps.copy()

DROP NHỮNG THUỘC TÍNH CÓ HƠN 50% LÀ NULL (TRỪ PRICE VÀ TẦN SỐ QUÉT)

In [316]:
thresh_limit = 0.5 * df.shape[0]

cols_to_check = [col for col in df.columns if col != "Tần số quét"]

cols_to_keep = (
    df[cols_to_check].dropna(thresh=thresh_limit, axis=1).columns.tolist()
)

cols_to_keep.append("Tần số quét")

df = df[cols_to_keep]

In [317]:
df.columns

Index(['Name', 'Link', 'Kích thước màn hình', 'Công nghệ màn hình',
       'Camera sau', 'Camera trước', 'Chipset', 'Công nghệ NFC',
       'Bộ nhớ trong', 'Thẻ SIM', 'Hệ điều hành', 'Độ phân giải màn hình',
       'Tính năng màn hình', 'Loại CPU', 'Dung lượng RAM', 'Pin',
       'Tần số quét'],
      dtype='str')

 ĐỐI TÊN THUỘC TÍNH 

In [318]:
feature_mapping = {
    "Tên": "Name",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
}

df = df.rename(columns=feature_mapping)


In [319]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               966 non-null    str  
 1   Link               966 non-null    str  
 2   Screen Size        873 non-null    str  
 3   Display            814 non-null    str  
 4   Rear Camera        855 non-null    str  
 5   Front Camera       823 non-null    str  
 6   Chipset            857 non-null    str  
 7   NFC                762 non-null    str  
 8   ROM                915 non-null    str  
 9   SIM Card           696 non-null    str  
 10  Operating System   762 non-null    str  
 11  Screen Resolution  659 non-null    str  
 12  Display Features   732 non-null    str  
 13  CPU                588 non-null    str  
 14  RAM                870 non-null    str  
 15  Battery            869 non-null    str  
 16  Tần số quét        206 non-null    str  
dtypes: str(17)
memory usage: 12

IMPORT DATA TỪ ANTUTU

In [320]:
antutu = pd.read_csv(r'antutu_socket.csv')
att = antutu.copy()

LẤY THÊM DATA TỪ GSM

In [321]:
gsm = pd.read_csv(r'all_phones_final.csv')
mem = gsm[['name_clean', 'Memory | Internal']]
ref_rate = gsm[['name_clean', 'Display | Type']]
battery = gsm[['name_clean', 'Battery | Type']]

### CLEANING PROCESS

CLEAN NAME

In [322]:
df["Name"] = df["Name"].apply(cln.clean_phone_name)
mem['name_clean'] = mem['name_clean'].apply(cln.clean_phone_name)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.clean_phone_name)
battery['name_clean'] = battery['name_clean'].apply(cln.clean_phone_name)

CLEAN CHIPSET AND ADD CHIPSET INFO TỪ ANTUTU

In [323]:
df['Chipset'] = df['Chipset'].apply(cln.clean_chipset)
df = cln.add_chipset_info(att, df)

In [324]:
specified_features = ['antutu_11', 'clock', 'gpu', 'architecture'] 

non_null_ratios = df[specified_features].notna().mean()

coverage_df = pd.DataFrame({
    'Non-Null Count': df[specified_features].notna().sum(),
    'Coverage Ratio (%)': (non_null_ratios * 100).round(2)
})

display(coverage_df)

,Non-Null Count,Coverage Ratio (%)
antutu_11,749,77.54
clock,749,77.54
gpu,749,77.54
architecture,749,77.54


CLEAN OPERATING SYSTEM

In [325]:
df["OS_Name"], \
    df["OS_Version"] = zip(*df["Operating System"].apply(cln.advanced_clean_os))

CLEAN NFC

In [326]:
df['NFC'] = df['NFC'].map(lambda x : 1 if x == "Có" else 0)

### EXTRACTING PROCESS

EXTRACT BRAND

In [327]:
df['Brand'] = df['Name'].apply(cln.get_brand)
df[['Name','Brand']].head()

,Name,Brand
0,iphone 17 pro,apple
1,oppo find x9s,oppo
2,samsung galaxy s26 ultra,samsung
3,iphone 17 pro max,apple
4,samsung galaxy s26,samsung


EXTRACT METRICS

In [328]:
cols_to_clean = ["Screen Size", "clock"]
for col in cols_to_clean:
    df[col] = df[col].apply(cln.clean_metrics)

In [329]:
df[["Screen Size", "clock"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Screen Size  873 non-null    float64
 1   clock        749 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


EXTRACT REFRESH RATE

In [330]:
df['Refresh Rate'] = df.apply(
    lambda row: cln.extract_refresh_rate(row['Tần số quét'], row['Display Features']),
    axis=1
)

In [331]:
df["Refresh Rate"].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Refresh Rate
Non-Null Count  Dtype  
--------------  -----  
633 non-null    float64
dtypes: float64(1)
memory usage: 7.7 KB


ADD RAM

In [332]:
df = cln.add_ram(mem, df)

In [333]:
df['RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: RAM
Non-Null Count  Dtype
--------------  -----
896 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB


EXTRACT STORAGE

In [334]:
df["RAM"] = df["RAM"].apply(cln.clean_storage)
df["ROM"] = df["ROM"].apply(cln.clean_storage)

In [335]:
df[["RAM", "ROM"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RAM     896 non-null    float64
 1   ROM     915 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


EXTRACT BATTERY

In [336]:
df['Battery'] = df['Battery'].apply(cln.clean_battery)
df = cln.fill_iphone_battery(df)

In [337]:
battery["Battery"] = battery["Battery | Type"].apply(cln.extract_battery_gsm)
battery.drop(columns= 'Battery | Type', inplace=True)

In [338]:
battery_clean = battery[['name_clean', 'Battery']].drop_duplicates(subset=['name_clean'])

df_merged = pd.merge(
    df,
    battery_clean[["name_clean", "Battery"]],  
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  # Đuôi phân biệt nếu trùng tên cột
)

df_merged["Battery"] = df_merged["Battery"].fillna(
    df_merged["Battery_from_ref"]
)

df_merged.drop(columns=["name_clean", "Battery_from_ref"], inplace=True)

df = df_merged

EXTRACT CPU


In [339]:
df[["total_cores", "max_freq_ghz", "min_freq_ghz", "weighted_mean"]] = df["architecture"].apply(
    lambda x: pd.Series(cln.extract_architecture(x))
)

EXTRACT RESOLUTION

In [340]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(cln.extract_res_row)
)

EXTRACT SIM


In [341]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(cln.clean_sim_options))

EXTRACT CAMERA

In [342]:
df = cln.extract_camera_info(df)

EXTRACT REFRESH RATE TỪ GSM VÀ MERGE VỚI DỮ LIỆU TỪ CPS


In [343]:
ref_rate["Refresh Rate"] = ref_rate["Display | Type"].apply(cln.extract_hz_gsm)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.remove_first_word_if_iphone)
ref_rate.drop(columns= 'Display | Type', inplace=True)

In [344]:
ref_rate_clean = ref_rate[['name_clean', 'Refresh Rate']].drop_duplicates(subset=['name_clean'])

df_merged = pd.merge(
    df,
    ref_rate_clean[["name_clean", "Refresh Rate"]],  # Chỉ lấy 2 cột cần thiết
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  # Đuôi phân biệt nếu trùng tên cột
)

df_merged["Refresh Rate"] = df_merged["Refresh Rate"].fillna(
    df_merged["Refresh Rate_from_ref"]
)

df_merged.drop(columns=["name_clean", "Refresh Rate_from_ref"], inplace=True)

df = df_merged

EXTRACT DISPLAY

In [345]:

df['Display'] = df['Display'].apply(cln.extract_display_type)

EXTRACT CHIPSET

In [346]:
df['Chipset_name'] = df['Chipset'].apply(cln.extract_chipset)

In [347]:
df['Chipset_gen'] = df['Chipset'].apply(cln.extract_chipset_gen)

EXTRACT GPU

In [348]:
df['gpu_name'] = df['gpu'].apply(cln.extract_gpu)

In [349]:
df['gpu_gen'] = df['gpu'].apply(cln.extract_gpu_gen)

### LOẠI BỎ NHỮNG DÒNG KHÔNG PHẢI LÀ ĐIỆN THOẠI

In [350]:
mask_not_smartphone = (
    (df['Screen Size'] < 4.0) |
    (df['Name'].str.lower().str.contains('tab|pad|win rt', na=False))
)

df = df[~mask_not_smartphone].reset_index(drop=True)
print(f'Đã drop {mask_not_smartphone.sum()} sản phẩm, còn lại {len(df)}')

Đã drop 42 sản phẩm, còn lại 924


### DERIVING PROCESS

Derive PPI

In [351]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)

df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

df.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)


Derive SIM_total

In [352]:
df['SIM_total'] = (
    df['Nano_SIM_Count'] +
    df['eSIM_Count'] +
    df['Micro_SIM_Count'] +
    df['Mini_SIM_Count']
).clip(upper=2)

df.drop(columns=[ 'Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count'], inplace=True)

Derive eSIM, dien thoai nao co eSIM: 1, khong co: 0

In [353]:
df['has_eSIM'] = (df['eSIM_Count'] > 0).astype(int)

df.drop(columns=['eSIM_Count'], inplace=True)

In [354]:
drop = [
    'Rear Camera', 'Front Camera', 'Screen Resolution', 'Display Features',
    'Operating System', 'SIM Card', 'OS_Is_Android', 'Tần số quét',
    'CPU', 'architecture', 'camera_score', 'Link']

df.drop(columns=drop, inplace = True, errors='ignore')

### TÍNH TỈ LỆ NULL CÒN LẠI

In [355]:
# Tính tỷ lệ % null của từng cột
null_percentages_before = (df.isnull().sum() / df.shape[0]) * 100

# Chỉ lọc ra những cột có % null > 0
null_cols_percentage_before = null_percentages_before[null_percentages_before > 0].sort_values(ascending=False)

print("Tỷ lệ % null theo từng cột:")
print(null_cols_percentage_before.round(2).astype(str) + '%')

Tỷ lệ % null theo từng cột:
front_f/          48.38%
PPI               31.49%
OS_Version        27.92%
gpu               21.43%
clock             21.43%
antutu_11         21.43%
max_freq_ghz      21.43%
weighted_mean     21.43%
min_freq_ghz      21.43%
Refresh Rate      21.43%
total_cores       21.43%
front_mp          12.12%
Chipset           11.69%
Screen Size        9.96%
rear_count         9.52%
rear_telephoto     9.42%
rear_ois           9.42%
rear_f/            9.42%
rear_mp_max        9.42%
rear_wide          9.42%
Battery            6.17%
RAM                 4.0%
ROM                 3.9%
dtype: str


### LỌC RA NHỮNG ĐIỆN THOẠI CÓ ĐỦ DATA VỀ ANTUTU SCORE


In [356]:
df = df[
    df['antutu_11'].notna()  
].copy()

print(len(df))  

726


In [357]:
# Tính tỷ lệ % null của từng cột
null_percentages_after = (df.isnull().sum() / df.shape[0]) * 100

# Chỉ lọc ra những cột có % null > 0
null_cols_percentage_after = null_percentages_after[null_percentages_after > 0].sort_values(ascending=False)

print("Tỷ lệ % null theo từng cột:")
print(null_cols_percentage_after.round(2).astype(str) + '%')

Tỷ lệ % null theo từng cột:
front_f/          48.35%
PPI               35.95%
OS_Version        29.75%
Refresh Rate       20.8%
front_mp          15.15%
Chipset           14.88%
Screen Size       12.53%
rear_count        11.98%
rear_telephoto    11.85%
rear_ois          11.85%
rear_f/           11.85%
rear_mp_max       11.85%
rear_wide         11.85%
Battery            7.85%
ROM                4.82%
RAM                4.68%
dtype: str


### FILLING MISSING VALUES 

Fill các features camera dựa vào Chipset


In [358]:
df = cln.fill_camera_by_chipset(df)

Fill RAM theo chipset

In [359]:
df = cln.fill_iphone_ram(df)
df = cln.fill_ram_by_chipset(df)

Fill ROM theo Chipset

In [360]:
df = cln.fill_rom_by_chipset(df)

Fill Battery theo RAM

In [361]:
df = cln.fill_battery_by_ram(df)

Fill Screen Size

In [362]:
df = cln.fill_screen_size(df)

Fill PPI

In [363]:
df = cln.fill_ppi(df)

Fill Refresh Rate

In [364]:
df = cln.fill_refresh_rate(df)

In [365]:
drop_cols = ['gpu', 'Chipset']

df.drop(columns = drop_cols, inplace=True)

In [366]:
df.info()

<class 'pandas.DataFrame'>
Index: 726 entries, 0 to 923
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            726 non-null    str    
 1   Screen Size     726 non-null    float64
 2   Display         726 non-null    str    
 3   NFC             726 non-null    int64  
 4   ROM             726 non-null    float64
 5   RAM             726 non-null    float64
 6   Battery         726 non-null    float64
 7   antutu_11       726 non-null    float64
 8   clock           726 non-null    float64
 9   OS_Name         726 non-null    str    
 10  OS_Version      510 non-null    float64
 11  Brand           726 non-null    str    
 12  Refresh Rate    726 non-null    float64
 13  total_cores     726 non-null    float64
 14  max_freq_ghz    726 non-null    float64
 15  min_freq_ghz    726 non-null    float64
 16  weighted_mean   726 non-null    float64
 17  rear_count      726 non-null    float64
 18  rear_m

In [367]:
df.head()

,Name,Screen Size,Display,NFC,ROM,RAM,Battery,antutu_11,clock,OS_Name,...,rear_wide,front_mp,front_f/,Chipset_name,Chipset_gen,gpu_name,gpu_gen,PPI,SIM_total,has_eSIM
0,iphone 17 pro,6.30,Other,1,256.0,12.0,3274.0,2606807.0,4260.0,iOS,...,1.0,18.0,1.9,Apple,A19 Pro,Apple GPU,A19 Pro,458.1,2,0
1,oppo find x9s,6.59,AMOLED,1,256.0,12.0,7025.0,3041307.0,3730.0,Android,...,1.0,32.0,2.2,MediaTek,9500S,Mali,G925,460.1,2,1
2,samsung galaxy s26 ultra,6.90,AMOLED,1,256.0,12.0,5000.0,3932243.0,4610.0,Android,...,1.0,12.0,2.2,Snapdragon,8 Elite Gen 5,Adreno,840,498.0,2,1
3,iphone 17 pro max,6.90,Other,1,256.0,12.0,3274.0,2606807.0,4260.0,iOS,...,1.0,18.0,1.9,Apple,A19 Pro,Apple GPU,A19 Pro,457.6,2,0
4,samsung galaxy s26,6.30,AMOLED,1,256.0,12.0,4300.0,3145925.0,3800.0,Android,...,1.0,12.0,2.2,Exynos,2600,Xclipse,960,409.1,2,1


In [368]:
mask_not_smartphone = (
    (df['Screen Size'] < 4.0) |          # feature phone
    (df['RAM'] < 1.0)         |          # feature phone
    (df['ROM'] < 1.0)         |          # feature phone
    (df['Name'].str.lower().str.contains('tab|pad|win rt', na=False))  # tablet
)

print(f'Số sản phẩm không phải smartphone: {mask_not_smartphone.sum()}')
print(df.loc[mask_not_smartphone, ['Name', 'Screen Size', 'RAM', 'ROM']].to_string())

Số sản phẩm không phải smartphone: 1
                Name  Screen Size  RAM   ROM
864  google pixel 8a          6.1  8.0  0.25


In [369]:
df.loc[df['Name'].str.strip().str.lower() == 'google pixel 8a', 'ROM'] = 256

In [370]:
df.info()

<class 'pandas.DataFrame'>
Index: 726 entries, 0 to 923
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            726 non-null    str    
 1   Screen Size     726 non-null    float64
 2   Display         726 non-null    str    
 3   NFC             726 non-null    int64  
 4   ROM             726 non-null    float64
 5   RAM             726 non-null    float64
 6   Battery         726 non-null    float64
 7   antutu_11       726 non-null    float64
 8   clock           726 non-null    float64
 9   OS_Name         726 non-null    str    
 10  OS_Version      510 non-null    float64
 11  Brand           726 non-null    str    
 12  Refresh Rate    726 non-null    float64
 13  total_cores     726 non-null    float64
 14  max_freq_ghz    726 non-null    float64
 15  min_freq_ghz    726 non-null    float64
 16  weighted_mean   726 non-null    float64
 17  rear_count      726 non-null    float64
 18  rear_m

In [371]:
# This shows every row that has a duplicate name
print(df[df.duplicated(subset=['Name'], keep=False)])

                         Name  Screen Size Display  NFC    ROM   RAM  Battery  \
0               iphone 17 pro         6.30   Other    1  256.0  12.0   3274.0   
2    samsung galaxy s26 ultra         6.90  AMOLED    1  256.0  12.0   5000.0   
3           iphone 17 pro max         6.90   Other    1  256.0  12.0   3274.0   
4          samsung galaxy s26         6.30  AMOLED    1  256.0  12.0   4300.0   
5                   iphone 17         6.30    OLED    1  256.0   8.0   3274.0   
..                        ...          ...     ...  ...    ...   ...      ...   
908                 realme 10         6.40  AMOLED    1  256.0   8.0   5000.0   
909           xiaomi 14 ultra         6.73  AMOLED    1  256.0  12.0   5300.0   
915                  honor 90         6.70  AMOLED    1  256.0   8.0   4900.0   
916                oneplus 10         6.67   Other    1  256.0   8.0   5000.0   
917           oneplus nord 2t         6.43  AMOLED    1  192.0   6.0   4500.0   

     antutu_11   clock  OS_

In [372]:
a = df[df['Name'] == 'iphone 17']
a

,Name,Screen Size,Display,NFC,ROM,RAM,Battery,antutu_11,clock,OS_Name,...,rear_wide,front_mp,front_f/,Chipset_name,Chipset_gen,gpu_name,gpu_gen,PPI,SIM_total,has_eSIM
5,iphone 17,6.3,OLED,1,256.0,8.0,3274.0,2239708.0,4260.0,iOS,...,1.0,18.0,1.9,Apple,A19,Apple GPU,A19,458.1,2,0
58,iphone 17,6.3,OLED,1,512.0,8.0,3274.0,2239708.0,4260.0,iOS,...,1.0,18.0,1.9,Apple,A19,Apple GPU,A19,458.1,2,0


In [373]:
df= df.groupby('Name').agg(
    **{col: (col, 'first') for col in df.columns if col not in ['Name', 'RAM', 'ROM']},
    RAM_min=('RAM', 'min'),
    RAM_max=('RAM', 'max'),
    ROM_min=('ROM', 'min'),
    ROM_max=('ROM', 'max'),
).reset_index()

In [374]:
df = cln.fill_missing_os_by_brand(
    df, brand_col="Brand", os_col="OS_Version"
)

In [375]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Data columns (total 34 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            578 non-null    str    
 1   Screen Size     578 non-null    float64
 2   Display         578 non-null    str    
 3   NFC             578 non-null    int64  
 4   Battery         578 non-null    float64
 5   antutu_11       578 non-null    float64
 6   clock           578 non-null    float64
 7   OS_Name         578 non-null    str    
 8   OS_Version      578 non-null    float64
 9   Brand           578 non-null    str    
 10  Refresh Rate    578 non-null    float64
 11  total_cores     578 non-null    float64
 12  max_freq_ghz    578 non-null    float64
 13  min_freq_ghz    578 non-null    float64
 14  weighted_mean   578 non-null    float64
 15  rear_count      578 non-null    float64
 16  rear_mp_max     578 non-null    float64
 17  rear_f/         578 non-null    float64
 18  r

In [377]:
df.to_csv('preprocess_1.csv')